# Лабораторная работа №5

## Цель работы
Освоить практические навыки работы с реляционными базами данных в Python, научиться создавать таблицы, выполнять операции добавления, чтения, обновления и удаления данных (CRUD), работать с транзакциями и обрабатывать ошибки.

## Задание

### Часть 1: Создание базы данных библиотеки (40 баллов)

Создайте базу данных `library.db` для учёта книг в библиотеке.

**Требования к таблице `books`:**
- `id` — уникальный идентификатор книги (INTEGER, PRIMARY KEY, AUTOINCREMENT)
- `title` — название книги (TEXT, NOT NULL)
- `author` — автор книги (TEXT, NOT NULL)
- `year` — год издания (INTEGER)
- `isbn` — международный стандартный книжный номер (TEXT, UNIQUE)
- `available` — доступна ли книга (INTEGER: 1 - да, 0 - нет, по умолчанию 1)

**Что нужно сделать:**
1. Создайте функцию `create_library_database()`, которая создаёт базу данных и таблицу
2. Напишите функцию `add_book(title, author, year, isbn)` для добавления новой книги
3. Добавьте в базу минимум 7 книг (используйте `executemany` для добавления нескольких книг одновременно)

### Часть 2: Функции для работы с данными (40 баллов)

Реализуйте следующие функции:

1. **`show_all_books()`** — вывести список всех книг в формате:
   ```
   ID: 1 | Название: "Война и мир" | Автор: Л.Н. Толстой | Год: 1869 | Доступна: Да
   ```

2. **`find_books_by_author(author_name)`** — найти все книги конкретного автора

3. **`find_books_by_year_range(start_year, end_year)`** — найти книги, изданные в указанном диапазоне лет

4. **`borrow_book(book_id)`** — отметить книгу как выданную (available = 0). Функция должна:
   - Проверить, существует ли книга с таким ID
   - Проверить, доступна ли книга (если available = 0, вывести сообщение "Книга уже выдана")
   - Обновить статус книги

5. **`return_book(book_id)`** — отметить книгу как возвращённую (available = 1)

6. **`delete_book(book_id)`** — удалить книгу из базы данных

### Часть 3: Статистика и обработка ошибок (20 баллов)

1. **`get_statistics()`** — вывести статистику:
   - Общее количество книг в библиотеке
   - Количество доступных книг
   - Количество выданных книг
   - Самый ранний и самый поздний год издания

2. Добавьте обработку ошибок во все функции:
   - Обработка `sqlite3.IntegrityError` (дублирование ISBN)
   - Обработка других ошибок базы данных
   - Проверка существования записей перед обновлением/удалением

### Часть 4: Главная программа

Создайте простое меню для взаимодействия с пользователем:

```
=== Система учёта библиотеки ===
1. Показать все книги
2. Добавить книгу
3. Найти книги по автору
4. Найти книги по годам
5. Выдать книгу
6. Вернуть книгу
7. Удалить книгу
8. Показать статистику
0. Выход
```

## Требования к оформлению

1. Код должен быть структурирован в функции
2. Используйте контекстный менеджер `with` для работы с базой
3. Обязательно используйте плейсхолдеры `?` для передачи параметров
4. Добавьте docstring к каждой функции

## Пример выполнения

```python
# Создание базы данных
create_library_database()

# Добавление книг
books_data = [
    ("Война и мир", "Л.Н. Толстой", 1869, "978-5-17-082549-6"),
    ("Преступление и наказание", "Ф.М. Достоевский", 1866, "978-5-17-082550-2"),
    # ... другие книги
]
add_multiple_books(books_data)

# Просмотр всех книг
show_all_books()

# Выдача книги
borrow_book(1)

# Статистика
get_statistics()
```

## Критерии оценки

- **Часть 1** (40 баллов): Корректное создание базы данных и таблицы, добавление книг
- **Часть 2** (40 баллов): Реализация всех функций для работы с данными
- **Часть 3** (20 баллов): Статистика и обработка ошибок

**Максимальный балл: 100**



In [ ]:
import sqlite3

DB_NAME = 'library.db'


def create_library_database():
    """Создаёт базу данных и таблицу books."""
    with sqlite3.connect(DB_NAME) as conn:
        conn.execute('''
            CREATE TABLE IF NOT EXISTS books (
                id        INTEGER PRIMARY KEY AUTOINCREMENT,
                title     TEXT    NOT NULL,
                author    TEXT    NOT NULL,
                year      INTEGER,
                isbn      TEXT    UNIQUE,
                available INTEGER DEFAULT 1
            )
        ''')
    print("База данных создана.")


def add_book(title, author, year, isbn):
    """Добавляет одну книгу в базу данных."""
    try:
        with sqlite3.connect(DB_NAME) as conn:
            conn.execute(
                'INSERT INTO books (title, author, year, isbn) VALUES (?, ?, ?, ?)',
                (title, author, year, isbn)
            )
        print(f'Книга "{title}" добавлена.')
    except sqlite3.IntegrityError:
        print(f'Ошибка: книга с ISBN {isbn} уже существует.')
    except sqlite3.Error as e:
        print(f'Ошибка базы данных: {e}')


def add_multiple_books(books_data):
    """Добавляет несколько книг через executemany."""
    try:
        with sqlite3.connect(DB_NAME) as conn:
            conn.executemany(
                'INSERT INTO books (title, author, year, isbn) VALUES (?, ?, ?, ?)',
                books_data
            )
        print(f'Добавлено {len(books_data)} книг.')
    except sqlite3.IntegrityError as e:
        print(f'Ошибка дублирования ISBN: {e}')
    except sqlite3.Error as e:
        print(f'Ошибка базы данных: {e}')


def show_all_books():
    """Выводит список всех книг."""
    with sqlite3.connect(DB_NAME) as conn:
        rows = conn.execute('SELECT id, title, author, year, available FROM books').fetchall()
    if not rows:
        print('Библиотека пуста.')
        return
    for row in rows:
        avail = 'Да' if row[4] else 'Нет'
        print(f'ID: {row[0]} | Название: "{row[1]}" | Автор: {row[2]} | Год: {row[3]} | Доступна: {avail}')


def find_books_by_author(author_name):
    """Находит все книги конкретного автора."""
    with sqlite3.connect(DB_NAME) as conn:
        rows = conn.execute(
            'SELECT id, title, author, year, available FROM books WHERE author LIKE ?',
            (f'%{author_name}%',)
        ).fetchall()
    if not rows:
        print(f'Книги автора "{author_name}" не найдены.')
    for row in rows:
        avail = 'Да' if row[4] else 'Нет'
        print(f'ID: {row[0]} | "{row[1]}" | {row[2]} | {row[3]} | Доступна: {avail}')


def find_books_by_year_range(start_year, end_year):
    """Находит книги, изданные в диапазоне лет."""
    with sqlite3.connect(DB_NAME) as conn:
        rows = conn.execute(
            'SELECT id, title, author, year FROM books WHERE year BETWEEN ? AND ?',
            (start_year, end_year)
        ).fetchall()
    for row in rows:
        print(f'ID: {row[0]} | "{row[1]}" | {row[2]} | {row[3]}')


def borrow_book(book_id):
    """Отмечает книгу как выданную."""
    try:
        with sqlite3.connect(DB_NAME) as conn:
            row = conn.execute('SELECT available FROM books WHERE id = ?', (book_id,)).fetchone()
            if row is None:
                print(f'Книга с ID {book_id} не найдена.')
                return
            if row[0] == 0:
                print('Книга уже выдана.')
                return
            conn.execute('UPDATE books SET available = 0 WHERE id = ?', (book_id,))
        print(f'Книга ID {book_id} выдана.')
    except sqlite3.Error as e:
        print(f'Ошибка: {e}')


def return_book(book_id):
    """Отмечает книгу как возвращённую."""
    try:
        with sqlite3.connect(DB_NAME) as conn:
            row = conn.execute('SELECT id FROM books WHERE id = ?', (book_id,)).fetchone()
            if row is None:
                print(f'Книга с ID {book_id} не найдена.')
                return
            conn.execute('UPDATE books SET available = 1 WHERE id = ?', (book_id,))
        print(f'Книга ID {book_id} возвращена.')
    except sqlite3.Error as e:
        print(f'Ошибка: {e}')


def delete_book(book_id):
    """Удаляет книгу из базы данных."""
    try:
        with sqlite3.connect(DB_NAME) as conn:
            row = conn.execute('SELECT id FROM books WHERE id = ?', (book_id,)).fetchone()
            if row is None:
                print(f'Книга с ID {book_id} не найдена.')
                return
            conn.execute('DELETE FROM books WHERE id = ?', (book_id,))
        print(f'Книга ID {book_id} удалена.')
    except sqlite3.Error as e:
        print(f'Ошибка: {e}')


def get_statistics():
    """Выводит статистику библиотеки."""
    with sqlite3.connect(DB_NAME) as conn:
        total = conn.execute('SELECT COUNT(*) FROM books').fetchone()[0]
        available = conn.execute('SELECT COUNT(*) FROM books WHERE available = 1').fetchone()[0]
        borrowed = conn.execute('SELECT COUNT(*) FROM books WHERE available = 0').fetchone()[0]
        years = conn.execute('SELECT MIN(year), MAX(year) FROM books').fetchone()
    print(f'Всего книг: {total}')
    print(f'Доступно: {available}')
    print(f'Выдано: {borrowed}')
    if years[0]:
        print(f'Годы изданий: от {years[0]} до {years[1]}')


def main():
    """Главное меню программы."""
    create_library_database()
    books_data = [
        ('Война и мир', 'Л.Н. Толстой', 1869, '978-5-17-082549-6'),
        ('Преступление и наказание', 'Ф.М. Достоевский', 1866, '978-5-17-082550-2'),
        ('Мастер и Маргарита', 'М.А. Булгаков', 1967, '978-5-17-082551-9'),
        ('Анна Каренина', 'Л.Н. Толстой', 1877, '978-5-17-082552-6'),
        ('Идиот', 'Ф.М. Достоевский', 1869, '978-5-17-082553-3'),
        ('Отцы и дети', 'И.С. Тургенев', 1862, '978-5-17-082554-0'),
        ('Евгений Онегин', 'А.С. Пушкин', 1833, '978-5-17-082555-7'),
    ]
    add_multiple_books(books_data)

    while True:
        print('\n=== Система учёта библиотеки ===')
        print('1. Показать все книги')
        print('2. Добавить книгу')
        print('3. Найти книги по автору')
        print('4. Найти книги по годам')
        print('5. Выдать книгу')
        print('6. Вернуть книгу')
        print('7. Удалить книгу')
        print('8. Показать статистику')
        print('0. Выход')

        choice = input('Выберите действие: ').strip()
        if choice == '1':
            show_all_books()
        elif choice == '2':
            title = input('Название: ')
            author = input('Автор: ')
            year = int(input('Год: '))
            isbn = input('ISBN: ')
            add_book(title, author, year, isbn)
        elif choice == '3':
            author = input('Имя автора: ')
            find_books_by_author(author)
        elif choice == '4':
            start = int(input('Начальный год: '))
            end = int(input('Конечный год: '))
            find_books_by_year_range(start, end)
        elif choice == '5':
            book_id = int(input('ID книги: '))
            borrow_book(book_id)
        elif choice == '6':
            book_id = int(input('ID книги: '))
            return_book(book_id)
        elif choice == '7':
            book_id = int(input('ID книги: '))
            delete_book(book_id)
        elif choice == '8':
            get_statistics()
        elif choice == '0':
            break
        else:
            print('Неверный выбор.')


# Демонстрация без интерактивного меню:
create_library_database()
books_data = [
    ('Война и мир', 'Л.Н. Толстой', 1869, '978-5-17-082549-6'),
    ('Преступление и наказание', 'Ф.М. Достоевский', 1866, '978-5-17-082550-2'),
    ('Мастер и Маргарита', 'М.А. Булгаков', 1967, '978-5-17-082551-9'),
    ('Анна Каренина', 'Л.Н. Толстой', 1877, '978-5-17-082552-6'),
    ('Идиот', 'Ф.М. Достоевский', 1869, '978-5-17-082553-3'),
    ('Отцы и дети', 'И.С. Тургенев', 1862, '978-5-17-082554-0'),
    ('Евгений Онегин', 'А.С. Пушкин', 1833, '978-5-17-082555-7'),
]
add_multiple_books(books_data)
show_all_books()
borrow_book(1)
get_statistics()
